# 推理资源账本与真实本机计时

浏览器 Python 实验：按顺序运行，变量在单元之间共享。图表来自当前代码计算。


## 1 · KV 显存账本

K、V 各一份；head 数指 KV heads。这里是估算，不是 GPU 显存采样。

In [ ]:
# The blog provides live charts; standalone Python prints chart data.
if 'display_plot' not in globals():
    def display_plot(x, y, title='', xlabel='', ylabel=''):
        print(title, list(zip(x,y)))

layers, kv_heads, head_dim = 32, 8, 128
bytes_per_element, concurrency = 2, 8
def kv_gib(tokens):
    return 2*layers*kv_heads*head_dim*bytes_per_element*concurrency*tokens/2**30
lengths = [512,1024,2048,4096,8192]
for n in lengths:
    print(n, "tokens =>", kv_gib(n), "GiB")
display_plot(lengths, [kv_gib(n) for n in lengths], "KV estimate", "tokens / request", "GiB")

## 2 · TP 分片真的保持结果吗

这个例子展示沿输入维度分片，局部乘积必须求和。输出维度分片则拼接结果，两者不同。

In [ ]:
x = [2,3,5,7]
W = [[1,2],[3,4],[5,6],[7,8]]
full = [sum(x[i]*W[i][j] for i in range(4)) for j in range(2)]
parts = [[sum(x[i]*W[i][j] for i in shard) for j in range(2)] for shard in ([0,1],[2,3])]
merged = [sum(p[j] for p in parts) for j in range(2)]
print("local contributions:",parts)
print("all-reduce sum:",merged, "reference:",full)
assert merged == full

## 3 · ZeRO 训练状态

16 bytes/参数 是此处混合精度 Adam 的约定：2 权重、2 梯度、4 master weight、8 moments；不含 activation。

In [ ]:
P, ranks = 7_000_000_000, 8
for stage in range(4):
    weights = 2*P/(ranks if stage>=3 else 1)
    gradients = 2*P/(ranks if stage>=2 else 1)
    optimizer = 12*P/(ranks if stage>=1 else 1)
    print("ZeRO",stage,"persistent GiB/rank:",(weights+gradients+optimizer)/2**30)
print("Peak also includes activations, temporary gathered parameters and buffers.")

## 4 · 真实浏览器 CPU 计时

测的是 Python 循环，不是 CUDA、BLAS 或 TPU。固定输入，先预热再重复；修改 n 观察增长。不要拿此结果对比推理框架。

In [ ]:
import time, statistics
n = 24
matrix = [[(i+j)%7/7 for j in range(n)] for i in range(n)]
def matmul():
    return [[sum(matrix[i][k]*matrix[k][j] for k in range(n)) for j in range(n)] for i in range(n)]
matmul()
times = []
for _ in range(12):
    start = time.perf_counter()
    out = matmul()
    times.append((time.perf_counter()-start)*1000)
print("shape:",(n,n),"checksum:",sum(map(sum,out)))
print("median ms:",statistics.median(times),"max ms:",max(times))
display_plot(list(range(1,13)),times,"Measured Python matmul","repeat","ms")